# ChromaDB Vector Store Explorer

Visualise et explore le contenu de la base de données ChromaDB contenant les documents Terraform.

In [1]:
from pathlib import Path
import pandas as pd
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
import sys

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))
from terraform_agent.config import Config

# Initialize configuration
config = Config(Path.cwd().parent)
vectorstore_path = config.PROJECT_ROOT / ".vectorstore"

print(f"📁 Looking for vectorstore at: {vectorstore_path}")
print(f"✓ Exists: {vectorstore_path.exists()}")

# Load vectorstore
embedding_function = OllamaEmbeddings(model=config.EMBEDDING_MODEL)
vectorstore = Chroma(
    collection_name="terraform_docs",
    embedding_function=embedding_function,
    persist_directory=str(vectorstore_path),
)

print("✓ Vectorstore loaded successfully")

# Get all documents
all_data = vectorstore.get()
ids = all_data["ids"]
metadatas = all_data["metadatas"]
documents = all_data["documents"]

print(f"📊 Total documents indexed: {len(ids)}")

# Create DataFrame for better visualization
data = []
for i, (doc_id, metadata, content) in enumerate(zip(ids, metadatas, documents)):
    data.append({
        'ID': doc_id,
        'Source': metadata.get('source', 'unknown'),
        'Content Preview': content[:80] + '...' if len(content) > 80 else content,
        'Length': len(content),
    })

df = pd.DataFrame(data)
print(df.to_string())

# Group by source
print("\n📚 Documents by Source:")
print(df.groupby('Source').size())

# Content length statistics
print("\n📈 Content Length Statistics:")
print(df['Length'].describe())

📁 Looking for vectorstore at: /Users/melkouhen/audit-tools/test-langchain/.vectorstore
✓ Exists: True
✓ Vectorstore loaded successfully
📊 Total documents indexed: 96
                                      ID                                                                           Source                                                                             Content Preview  Length
0   5dd84d76-2c3d-42ef-955a-a8902b7fd2af   /Users/melkouhen/audit-tools/test-langchain/rules/rules-tf-state-versioning.md       # Terraform Versioning & State Management Rules\n\n<rule id="TF-VERSION-PINNING-00...     789
1   fbb24ee3-2b09-4ea4-ba87-eaaf98ad083d   /Users/melkouhen/audit-tools/test-langchain/rules/rules-tf-state-versioning.md     <pattern id="correct">\n<title>✅ Pinned Versions</title>\n\n```hcl\n# versions.tf or...     983
2   d26908a1-03ce-4b9f-b583-574a8779d9b8   /Users/melkouhen/audit-tools/test-langchain/rules/rules-tf-state-versioning.md     <antipattern id="incorrect">\n<title>❌ Unp

## Search Documents

Recherche sémantique dans la base de données

In [2]:
# Interactive search
query = "module cloud storage"
k = 5

results = vectorstore.similarity_search(query, k=k)

print(f"🔍 Search Results for: '{query}'\n")
for i, result in enumerate(results, 1):
    print(f"{i}. [Source: {result.metadata.get('source', 'unknown')}]")
    print(f"   {result.page_content[:150]}...\n")

🔍 Search Results for: 'module cloud storage'

1. [Source: /Users/melkouhen/audit-tools/test-langchain/rules/rules-tf-project-structure.md]
   **Each module is:**
✓ Self-contained  
✓ Can be used independently  
✓ Requires minimal external knowledge  
</pattern>

<antipattern id="incorrect">
...

2. [Source: /Users/melkouhen/audit-tools/test-langchain/rules/rules-tf-project-structure.md]
   module "gcs" {
  source = "../../modules/gcs_bucket"
  bucket_name = var.bucket_name
  environment = var.environment
}
```

**EXAMPLE - Keep Inline (N...

3. [Source: /Users/melkouhen/audit-tools/test-langchain/rules/rules-tf-project-structure.md]
   <why>
**Shallow modules are easier to:**
1. Understand at a glance
2. Debug when things fail
3. Test independently
4. Navigate with clear file paths
5...

4. [Source: /Users/melkouhen/audit-tools/test-langchain/rules/rules-tf-naming-state.md]
   # Option 4: Destroy and recreate
terraform destroy -target=google_storage_bucket.archive
# Destroys resource i

In [3]:
# Advanced: Find documents about specific topics
topics = [
    "security",
    "modules",
    "variables",
    "state"
]

print("\n🎯 Topic-based Search Results:\n")
for topic in topics:
    results = vectorstore.similarity_search(topic, k=2)
    print(f"\n📌 {topic.upper()}:")
    if results:
        print(f"   {results[0].page_content[:150]}...")
    else:
        print("   No results found")


🎯 Topic-based Search Results:


📌 SECURITY:
   # ❌ WRONG: API key in code
variable "sendgrid_api_key" {
  default = "SG.xxxxxxxxxxxx"  # ❌ In version control forever
}

# ❌ WRONG: Service account k...

📌 MODULES:
   <why>
**Shallow modules are easier to:**
1. Understand at a glance
2. Debug when things fail
3. Test independently
4. Navigate with clear file paths
5...

📌 VARIABLES:
   <implementation-checklist>
- [ ] Identify all hardcoded values in code
- [ ] Create variables.tf with parameters
- [ ] Replace hardcoded values with v...

📌 STATE:
   **Local state is OK for:**
- Personal throwaway experiments
- Learning/tutorials
</when-to-apply>

<implementation-checklist>
- [ ] Choose backend ser...


In [4]:
# Summary
print("\n" + "="*60)
print("📊 VECTORSTORE SUMMARY")
print("="*60)
print(f"Total chunks indexed: {len(ids)}")
print(f"Unique source files: {df['Source'].nunique()}")
print(f"Average chunk size: {df['Length'].mean():.0f} characters")
print(f"Embedding model: {config.EMBEDDING_MODEL}")
print(f"Collection: terraform_docs")


📊 VECTORSTORE SUMMARY
Total chunks indexed: 96
Unique source files: 9
Average chunk size: 844 characters
Embedding model: nomic-embed-text
Collection: terraform_docs
